# Scope_Check — is every domain with content classified?

Release check for the consumer tracks. Run it after a CT or COSMoS refresh, before the
three Findings consumers.

The consumers take their scope from `SDTM_Domain_Metadata.xlsx`
(`Specimen_Based`, `Measurement`, `Instrument_Domain_Rules`) minus named exclusions
in each notebook. A domain that is missing from the metadata can therefore never
enter a consumer, and nothing would say so. This notebook makes that visible.

Every domain that carries content is listed:

- SDTM CT test codes (`SDTM_Test_Identity.xlsx`, `SDTM_Instrument_Test_Identity.xlsx`)
- COSMoS Dataset Specializations (`COSMoS_Graph.xlsx`, `DSS` sheet)

and checked against two lists:

- `SDTM_Domain_Metadata.xlsx` — the repo's own domain reference (public sources)
- the SDTM Domain Abbreviation codelist C66734 in the SDTM CT file — CDISC's published
  list of domain codes

**Fails** when a domain with content is missing from either list. A domain that is
classified but has no consumer class (e.g. SC, TU) is reported, not failed: that is a
decision recorded in the metadata.

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
INSTR_TEST_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Test_Identity.xlsx'
SDTM_CT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'downloads' / 'SDTM_Terminology.txt'
GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'

DOMAIN_CODELIST = 'C66734'            # SDTM Domain Abbreviation

for f in [DOMAIN_META_FILE, TEST_IDENTITY_FILE, INSTR_TEST_FILE, SDTM_CT_FILE, GRAPH_FILE]:
    if not f.exists():
        raise FileNotFoundError(f)
    print(f'  {f.relative_to(REPO_ROOT)}')

  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  sdtm-test-codes/machine_actionable/SDTM_Instrument_Test_Identity.xlsx
  sdtm-test-codes/downloads/SDTM_Terminology.txt
  cosmos-graph/interim/COSMoS_Graph.xlsx


## 2. Load

In [2]:
meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
instr_rules = pd.read_excel(DOMAIN_META_FILE, sheet_name='Instrument_Domain_Rules', dtype=str)
test_identity = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
instr_tests = pd.read_excel(INSTR_TEST_FILE, sheet_name='Test Codes', dtype=str).fillna('')
dss = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')
sdtm_ct = pd.read_csv(SDTM_CT_FILE, sep='\t', dtype=str)

ct_domains = sdtm_ct[sdtm_ct['Codelist Code'] == DOMAIN_CODELIST][['CDISC Submission Value', 'CDISC Synonym(s)']]
ct_domains = ct_domains.rename(columns={'CDISC Submission Value': 'Domain', 'CDISC Synonym(s)': 'CT_Domain_Name'})
if ct_domains.empty:
    raise ValueError(f'Codelist {DOMAIN_CODELIST} not found in {SDTM_CT_FILE.name}')

print(f'SDTM_Domain_Metadata domains:          {len(meta)}')
print(f'C66734 SDTM Domain Abbreviation terms: {len(ct_domains)}')

SDTM_Domain_Metadata domains:          67
C66734 SDTM Domain Abbreviation terms: 88


## 3. Domains with content

In [3]:
rows = []
for s in test_identity['SDTM_Domains']:
    for d in s.split(';'):
        if d.strip():
            rows.append(('CT_Test_Codes', d.strip()))
rows += [('CT_Instrument_Test_Codes', d) for d in instr_tests['Domain'] if d]
rows += [('COSMoS_DSS', d) for d in dss['domain'] if d]

content = pd.DataFrame(rows, columns=['source', 'Domain'])
content = content.groupby(['Domain', 'source']).size().unstack(fill_value=0).reset_index()
for col in ['CT_Test_Codes', 'CT_Instrument_Test_Codes', 'COSMoS_DSS']:
    if col not in content.columns:
        content[col] = 0

instrument_domains = set(instr_rules['Domain'])

def consumer_class(row):
    if row['Specimen_Based'] == 'Yes':
        return 'Specimen'
    if row['Measurement'] == 'Yes':
        return 'Measurement'
    if row['Domain'] in instrument_domains:
        return 'Instrument'
    return ''

meta['Consumer_Class'] = meta.apply(consumer_class, axis=1)

report = content.merge(meta[['Domain', 'Domain_Name', 'Observation_Class', 'Consumer_Class']],
                       on='Domain', how='left', indicator='in_meta')
report['In_Domain_Metadata'] = (report['in_meta'] == 'both').map({True: 'Yes', False: 'No'})
report = report.drop(columns='in_meta').merge(ct_domains, on='Domain', how='left')
report['In_CT_C66734'] = report['CT_Domain_Name'].notna().map({True: 'Yes', False: 'No'})
report = report.fillna('')
report = report[['Domain', 'Domain_Name', 'CT_Domain_Name', 'Observation_Class', 'Consumer_Class',
                 'CT_Test_Codes', 'CT_Instrument_Test_Codes', 'COSMoS_DSS',
                 'In_Domain_Metadata', 'In_CT_C66734']].sort_values('Domain').reset_index(drop=True)

with pd.option_context('display.max_rows', None, 'display.width', 200):
    print(report.to_string(index=False))

Domain                              Domain_Name                         CT_Domain_Name Observation_Class Consumer_Class  CT_Test_Codes  CT_Instrument_Test_Codes  COSMoS_DSS In_Domain_Metadata In_CT_C66734
    AE                           Adverse Events    Adverse Events; Adverse Experiences            Events                             0                         0           2                Yes          Yes
    AU                       Auricular Findings                     Auricular Findings          Findings                            10                         0           0                Yes          Yes
    BE                       Biospecimen Events                     Biospecimen Events            Events                             0                         0           7                Yes          Yes
    BS                     Biospecimen Findings                   Biospecimen Findings          Findings       Specimen             30                         0           0        

## 4. Result

In [4]:
no_class = report[(report['In_Domain_Metadata'] == 'Yes') & (report['Consumer_Class'] == '')]
print(f'Classified, no consumer class (reported, not failed): {", ".join(no_class["Domain"])}')

not_in_meta = report[report['In_Domain_Metadata'] == 'No']
not_in_ct = report[report['In_CT_C66734'] == 'No']

problems = []
if len(not_in_meta):
    problems.append(f'Not in SDTM_Domain_Metadata ({len(not_in_meta)}): {", ".join(not_in_meta["Domain"])}')
if len(not_in_ct):
    problems.append(f'Not a domain code in CT C66734 ({len(not_in_ct)}): {", ".join(not_in_ct["Domain"])}')

if problems:
    raise ValueError('Scope check FAILED\n  ' + '\n  '.join(problems))
print('Scope check PASSED — every domain with content is classified and is a published domain code.')

Classified, no consumer class (reported, not failed): AE, AU, BE, CM, DA, DD, DM, DO, DS, DU, EC, ED, EX, FA, GI, HM, IE, IG, MH, NV, OE, PR, PT, RP, SC, SR, SS, SU, TR, TS, TU
Scope check PASSED — every domain with content is classified and is a published domain code.
